# Baga AI — векторизация фото (SigLIP 2) на Kaggle

Считает эмбеддинги всех фото из `krisha.db` + типы комнат и отдаёт артефакты
в формате `krisha_search/artifacts`. Локальный поиск подхватывает их без правок.

**Почему SigLIP 2, а не DINOv3 из museum:** запрос текстовый («светлая кухня»), а
DINOv3 текста не понимает. **Почему не open_clip друга:** ViT-B-32 от OpenAI обучен на
английском, русские запросы понимает плохо.

**Из museum-сетапа:** `resize_and_pad` (вид `pad`), CLAHE (опция), `l2 → mean → l2`, чекпоинты.
**Не переносится:** мультискейл 336/512 — SigLIP обучена на фиксированном разрешении.

```
72 963 строк photos  ->  71 679 уникальных файлов (sha-дедуп друга)  ->  SigLIP 2  ->  photo_emb.npy
                                                        |
                             zero-shot: тип комнаты + «рендер или фото»  ->  photo_rooms.parquet
```

**Настройки Kaggle:** Accelerator **GPU T4**, Internet **ON**, датасет с `KrishaParser` добавлен в Input.
SigLIP 2 открытая — HF-токен не нужен. Base-модель: ~10 мин на всё.

⚠️ Препроцессинг ниже — **копия `krisha_search/embed.py`**. Меняешь здесь — меняй и там,
иначе векторы фото и запросов будут несравнимы (локальный код это проверит по `photo_emb.json`).

In [ ]:
!pip install -q -U "transformers>=4.56" sentencepiece

In [ ]:
import os, re, json, time, math, hashlib, sqlite3
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms as T
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# ---- настройки (должны совпадать с krisha_search/config.py)
MODEL_ID = "google/siglip2-base-patch16-224"   # качество: google/siglip2-so400m-patch14-384 (x4-5 дольше)
VIEWS = ("pad",)                               # ("pad", "full") — если validate robust покажет прирост
USE_CLAHE = False
BATCH = 64
WORKERS = os.cpu_count()
SHARD = 4096

OUT = Path("/kaggle/working/artifacts")       # -> krisha_search/artifacts
DATA_OUT = Path("/kaggle/working/data")       # -> krisha_search/data
CKPT = Path("/kaggle/working/emb_shards")
for d in (OUT, DATA_OUT, CKPT):
    d.mkdir(parents=True, exist_ok=True)

IMG_EXT = {".jpg", ".jpeg", ".png", ".webp"}

def find_photos_root(root="/kaggle/input"):
    """Папка, внутри которой лежат папки-объявления с числовыми именами."""
    for dp, dn, _ in os.walk(root):
        if Path(dp).name == "photos" and sum(d.isdigit() for d in dn) > 0:
            return Path(dp)
    raise FileNotFoundError(f"папка photos/<listing_id>/ не найдена в {root} — добавь датасет в Input")

def find_db(root="/kaggle/input"):
    """krisha.db не обязателен: без него эмбеддим по папкам, но не будет цен и описаний."""
    hits = [Path(dp) / "krisha.db" for dp, _, fn in os.walk(root) if "krisha.db" in fn]
    return hits[0] if hits else None

PHOTOS_ROOT = find_photos_root()
DB = find_db()

device = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if device == "cuda" else torch.float32
print("photos:", PHOTOS_ROOT, "\nDB:", DB or "нет — только фото, без цен и описаний")
print("device:", device, "| views:", VIEWS, "| clahe:", USE_CLAHE)
if device == "cpu":
    print("\n!!! GPU НЕ ВКЛЮЧЁН. На процессоре 71 679 фото считаются часами."
          "\n!!! Session options -> Accelerator -> GPU T4 x2, затем перезапустить.")

## 1. Индекс фото из базы

В базе пути виндовые (`C:\Users\...\photos\<id>\<n>.jpg`) — берём хвост `<id>/<n>.jpg`
относительно папки `photos`. Так индекс одинаково работает на Kaggle и локально.

Друг дедуплицировал фото по sha256: одинаковое фото у двух объявлений лежит одним
файлом (191 случай). Считаем каждый **файл** один раз, потом раздаём вектор всем строкам.

In [ ]:
LISTING_COLS = ["id", "url", "deal_type", "rent_period", "price_kzt", "rooms", "area_total",
                "area_living", "area_kitchen", "floor", "floors_total", "year_built", "building_type",
                "complex_name", "city", "district", "address", "lat", "lon", "description",
                "furniture", "bathroom", "balcony", "renovation_text", "published_at",
                "photos_count", "duplicate_group_id"]
REL = re.compile(r"(\d+)[\\/]+(\d+\.jpg)$")

def index_from_db(db_path):
    """Связка из базы: ловит и 191 фото, сохранённое в папке ДРУГОГО объявления (sha-дедуп)."""
    con = sqlite3.connect(f"file:{db_path}?mode=ro&immutable=1", uri=True)   # Input read-only
    listings = pd.read_sql(f"SELECT {', '.join(LISTING_COLS)} FROM listings", con)
    photos = pd.read_sql("SELECT listing_id, idx, local_path FROM photos WHERE local_path IS NOT NULL", con)
    print("объявлений:", len(listings), "| строк photos:", len(photos))
    print(listings.groupby(["deal_type", "rent_period", "city"]).size().to_string())

    def rel_path(p):   # пути в базе виндовые: C:\...\photos\<id>\<n>.jpg -> <id>/<n>.jpg
        m = REL.search(p or "")
        return f"{m[1]}/{m[2]}" if m else None

    photos["path"] = photos.local_path.map(rel_path)
    bad = photos.path.isna() | ~photos.path.map(lambda p: p is not None and (PHOTOS_ROOT / p).exists())
    print("путь не распознан или файла нет:", int(bad.sum()))
    idx = (photos[~bad].rename(columns={"idx": "photo_n"})
           .assign(listing_id=lambda d: d.listing_id.astype(str))[["listing_id", "photo_n", "path"]])
    return idx, listings

def index_from_folders(root):
    """Запасной режим: имя папки = listing_id. Проще, но теряет фото из чужих папок."""
    rows = []
    for d in sorted(p for p in root.iterdir() if p.is_dir()):
        for n, f in enumerate(sorted(x for x in d.iterdir() if x.suffix.lower() in IMG_EXT)):
            rows.append({"listing_id": d.name,
                         "photo_n": int(f.stem) if f.stem.isdigit() else n,
                         "path": f"{d.name}/{f.name}"})
    return pd.DataFrame(rows), None

idx, listings = index_from_db(DB) if DB else index_from_folders(PHOTOS_ROOT)
idx = idx.sort_values(["listing_id", "photo_n"]).reset_index(drop=True)
uniq = idx.path.unique()
row2file = pd.Index(uniq).get_indexer(idx.path)
print(f"\nстрок в индексе: {len(idx)} | уникальных файлов: {len(uniq)} | объявлений: {idx.listing_id.nunique()}")

## 2. Препроцессинг — копия `krisha_search/embed.py`

In [ ]:
BICUBIC = T.InterpolationMode.BICUBIC

def open_rgb(src, max_side=None):
    """EXIF-поворот обязателен: фото с телефона без него лежат на боку."""
    if isinstance(src, Image.Image):
        im = ImageOps.exif_transpose(src).convert("RGB")
    else:
        with Image.open(src) as raw:
            im = ImageOps.exif_transpose(raw).convert("RGB")
    if max_side:
        im.thumbnail((max_side, max_side))
    return im

def apply_clahe(im, clip_limit=2.0, tile=(8, 8)):
    """CLAHE только по яркости (L в LAB) — цвета ремонта не трогаем."""
    lab = cv2.cvtColor(np.asarray(im), cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile).apply(l)
    return Image.fromarray(cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB))

def preprocess(im):
    return apply_clahe(im) if USE_CLAHE else im

class PadToSquare:
    """resize_and_pad из museum-dorewka. Цвет полей = mean модели:
    после Normalize поля становятся нулями."""
    def __init__(self, size, fill):
        self.size, self.fill = size, fill

    def __call__(self, im):
        w, h = im.size
        k = self.size / max(w, h)
        nw, nh = max(1, round(w * k)), max(1, round(h * k))
        canvas = Image.new("RGB", (self.size, self.size), self.fill)
        canvas.paste(im.resize((nw, nh), Image.BICUBIC), ((self.size - nw) // 2, (self.size - nh) // 2))
        return canvas

def make_views(ip, views=VIEWS):
    size = ip.size if isinstance(ip.size, dict) else {}
    s = size.get("height") or size.get("shortest_edge") or 224
    norm = T.Normalize(ip.image_mean, ip.image_std)
    fill = tuple(round(m * 255) for m in ip.image_mean)
    specs = {
        "pad": T.Compose([PadToSquare(s, fill), T.ToTensor(), norm]),
        "full": T.Compose([T.Resize((s, s), interpolation=BICUBIC), T.ToTensor(), norm]),
        "center": T.Compose([T.Resize(s, interpolation=BICUBIC), T.CenterCrop(s), T.ToTensor(), norm]),
    }
    return {k: specs[k] for k in views}, s

class PhotoDS(Dataset):
    """Один декод файла -> все виды. Битое фото не выкидываем, а помечаем."""
    def __init__(self, paths, views, size):
        self.paths, self.views, self.size = paths, views, size

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            im = preprocess(open_rgb(self.paths[i]))
            return {k: tf(im) for k, tf in self.views.items()}, i, True
        except Exception:
            return {k: torch.zeros(3, self.size, self.size) for k in self.views}, i, False

## 3. Модель + проверка на одном батче

In [ ]:
from transformers import AutoModel, AutoProcessor

proc = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).eval().to(device, dtype=DTYPE)
views, S = make_views(proc.image_processor)

def _as_tensor(out):
    return out if torch.is_tensor(out) else out.pooler_output

@torch.inference_mode()
def encode(xs):
    """{вид: (B,3,S,S)} -> (B, D): нормировать каждый вид, сложить, снова нормировать."""
    acc = None
    for x in xs.values():
        v = F.normalize(_as_tensor(model.get_image_features(pixel_values=x.to(device, DTYPE))).float(), dim=-1)
        acc = v if acc is None else acc + v
    return F.normalize(acc, dim=-1).cpu().numpy()

@torch.inference_mode()
def embed_texts(texts):
    inp = proc(text=[t.lower() for t in texts], padding="max_length", max_length=64,
               truncation=True, return_tensors="pt").to(device)
    return F.normalize(_as_tensor(model.get_text_features(**inp)).float(), dim=-1).cpu().numpy()


files = [str(PHOTOS_ROOT / p) for p in uniq]


def benchmark(n=100, batch=BATCH, workers=WORKERS):
    """Замер на n фото: отдельно чтение с диска и отдельно GPU.

    Они работают параллельно (воркеры готовят следующий батч, пока GPU считает текущий),
    поэтому реальное время ближе к максимуму из двух, а не к сумме. Узкое место — то,
    что больше: если чтение, поможет workers/batch, если GPU — только модель поменьше.
    """
    sample = files[:n]
    dl = DataLoader(PhotoDS(sample, views, S), batch_size=batch, shuffle=False,
                    num_workers=workers, pin_memory=(device == "cuda"))

    t0 = time.time()
    batches = [xs for xs, _, _ in dl]          # только чтение + препроцессинг
    t_io = time.time() - t0

    v = encode(batches[0])                      # прогрев: первый вызов всегда дольше
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for xs in batches:
        v = encode(xs)
    if device == "cuda":
        torch.cuda.synchronize()
    t_gpu = time.time() - t0

    total = len(files)
    io_rate, gpu_rate = n / t_io, n / t_gpu
    best, worst = max(t_io, t_gpu) * total / n / 60, (t_io + t_gpu) * total / n / 60
    print(f"фото в замере: {n} | dim: {v.shape[1]} | размер входа: {S}px | device: {device}")
    print(f"  чтение с диска: {t_io:6.1f} с  ({io_rate:6.1f} фото/с, workers={workers})")
    print(f"  модель:         {t_gpu:6.1f} с  ({gpu_rate:6.1f} фото/с, batch={batch})")
    print(f"  узкое место:    {'чтение с диска' if t_io > t_gpu else 'модель'}")
    print(f"\nпрогноз на {total} фото: {best:.0f}-{worst:.0f} минут")
    return v.shape[1]


D = benchmark(100)

## 4. Эмбеддинги всех фото (шардами)

Каждый шард сразу на диске. Если сессия упала посреди — перезапусти ячейку, готовые шарды
пропустятся. (При «Save & Run All» `/kaggle/working` пустой, всё считается заново.)

In [ ]:
n_sh = math.ceil(len(files) / SHARD)
for s in tqdm(range(n_sh), desc="шарды"):
    f = CKPT / f"shard_{s:05d}.npz"
    if f.exists():
        continue
    chunk = files[s * SHARD:(s + 1) * SHARD]
    dl = DataLoader(PhotoDS(chunk, views, S), batch_size=BATCH, shuffle=False,
                    num_workers=WORKERS, pin_memory=True)
    E = np.zeros((len(chunk), D), dtype=np.float32)
    ok = np.zeros(len(chunk), dtype=bool)
    for xs, ii, good in tqdm(dl, leave=False):
        E[ii.numpy()] = encode(xs)
        ok[ii.numpy()] = good.numpy()
    E[~ok] = 0
    tmp = CKPT / f"shard_{s:05d}.tmp.npz"
    np.savez(tmp, emb=E.astype(np.float16), valid=ok)
    os.replace(tmp, f)

parts = [np.load(CKPT / f"shard_{s:05d}.npz") for s in range(n_sh)]
U = np.concatenate([p["emb"] for p in parts])
U_ok = np.concatenate([p["valid"] for p in parts])
assert len(U) == len(uniq)

E = U[row2file]            # вектор файла -> каждой строке индекса (sha-дубли получают одинаковый)
valid = U_ok[row2file]
print("эмбеддинги:", E.shape, "| битых файлов:", int((~U_ok).sum()))

## 5. Zero-shot: тип комнаты и «рендер или фото» — копия `krisha_search/rooms.py`

In [ ]:
ROOM_PROMPTS = {
    "kitchen":     ["фото кухни в квартире", "a photo of a kitchen in an apartment"],
    "living_room": ["фото гостиной комнаты", "a photo of a living room"],
    "bedroom":     ["фото спальни с кроватью", "a photo of a bedroom with a bed"],
    "bathroom":    ["фото ванной комнаты или санузла", "a photo of a bathroom with a toilet or bathtub"],
    "hallway":     ["фото коридора или прихожей в квартире", "a photo of an apartment hallway"],
    "balcony":     ["фото балкона или лоджии", "a photo of a balcony"],
    "exterior":    ["фасад жилого дома снаружи", "exterior of an apartment building"],
    "entrance":    ["подъезд или лестничная клетка", "a stairwell in an apartment building"],
    "floor_plan":  ["планировка квартиры, чертёж", "an apartment floor plan drawing"],
    "window_view": ["вид из окна на город или двор", "a view from the window"],
    "other":       ["документ или скриншот с текстом", "a screenshot with text"],
}
RENDER_PROMPTS = {
    "render": ["3d визуализация интерьера, рендер", "a 3d render of an interior, cgi visualization"],
    "photo":  ["обычное фото квартиры на телефон", "an amateur phone photo of a real apartment"],
}
INTERIOR = {"kitchen", "living_room", "bedroom", "bathroom", "hallway", "balcony"}
RENDER_THR = 0.7

def l2(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-12)

def classify(E, prompts):
    scale = float(model.logit_scale.exp()) if hasattr(model, "logit_scale") else 100.0
    names = list(prompts)
    C = np.stack([l2(embed_texts(prompts[n]).mean(0)) for n in names])
    logits = (E.astype(np.float32) @ C.T) * scale
    logits -= logits.max(1, keepdims=True)
    P = np.exp(logits)
    return names, P / P.sum(1, keepdims=True)

names, P = classify(E, ROOM_PROMPTS)
rnames, R = classify(E, RENDER_PROMPTS)
rooms = idx.copy()
rooms["room_type"] = np.array(names)[P.argmax(1)]
rooms["room_conf"] = P.max(1)
rooms["render_prob"] = R[:, rnames.index("render")]
rooms.loc[~valid, "room_type"] = "invalid"

print(rooms.room_type.value_counts().to_string())
print(f"\nпохоже на рендер: {(rooms.render_prob > RENDER_THR).mean():.1%} фото")

## 6. Сохранение артефактов (формат `krisha_search`)

In [ ]:
paths = idx.path.tolist()
meta = {"model": MODEL_ID, "views": list(VIEWS), "clahe": USE_CLAHE, "n": len(paths),
        "paths_hash": hashlib.sha1("\n".join(paths).encode()).hexdigest()}

idx.to_parquet(OUT / "photos.parquet", index=False)
np.save(OUT / "photo_emb.npy", E.astype(np.float16))
np.save(OUT / "photo_valid.npy", valid)
(OUT / "photo_emb.json").write_text(json.dumps(meta))
rooms.to_parquet(OUT / "photo_rooms.parquet", index=False)
if listings is not None:
    listings.to_parquet(DATA_OUT / "listings_raw.parquet", index=False)   # вход для `run_pipeline.py prepare`
else:
    print("базы не было -> без listings_raw.parquet: поиск по цене/описанию локально не заработает")

for p in sorted(OUT.iterdir()) + sorted(DATA_OUT.iterdir()):
    print(f"{p.stat().st_size / 1e6:8.1f} MB  {p}")

### Проверка связки id

Вектор в строке `i` относится к фото из строки `i` файла `photos.parquet`, а `listing_id`
оттуда стыкуется с объявлением в `listings_raw.parquet`. Ниже это видно целиком:
вектор -> фото -> объявление -> цена и описание.

In [ ]:
chk = pd.read_parquet(OUT / "photos.parquet")
E_chk, meta_chk = np.load(OUT / "photo_emb.npy"), json.loads((OUT / "photo_emb.json").read_text())

assert len(chk) == len(E_chk) == meta_chk["n"], "строк в индексе и векторов должно быть поровну"
assert meta_chk["paths_hash"] == hashlib.sha1("\n".join(chk.path).encode()).hexdigest(), "хэш путей не сошёлся"

# Имя папки — это НЕ всегда listing_id: парсер дедуплицирует фото по sha256, поэтому
# одинаковый файл хранится один раз, в папке первого объявления. Таких строк должно быть
# мало, и файл обязан встречаться в индексе ещё и у владельца папки.
owner = chk.path.str.split("/").str[0]
shared = chk[owner != chk.listing_id]
own_paths = set(chk.path[owner == chk.listing_id])
assert len(shared) / len(chk) < 0.01, f"слишком много фото из чужих папок: {len(shared)}"
orphan = int((~shared.path.isin(own_paths)).sum())

print(f"строк: {len(chk)} | векторов: {E_chk.shape} | объявлений: {chk.listing_id.nunique()}")
print(f"фото в папке другого объявления (sha-дедуп): {len(shared)}, из них без владельца: {orphan}")
print("норма первого вектора:", round(float(np.linalg.norm(E_chk[0].astype(np.float32))), 3), "(должна быть 1.0)")

if listings is not None:
    j = chk.merge(listings.assign(listing_id=lambda d: d.id.astype(str)), on="listing_id", how="left")
    assert j.price_kzt.notna().all(), "есть фото без объявления — связка по listing_id порвана"
    print()
    print(j.head(3)[["listing_id", "photo_n", "path", "price_kzt", "rooms", "city"]].to_string(index=False))
    print("\nописание для проверки:", str(j.description.iloc[0])[:160])

## 8. Посмотреть глазами — обязательно

Как в museum: цифры выше ничего не значат, пока не видно, что модель находит.
Смотрим: (1) правильно ли определены комнаты, (2) что считается рендером,
(3) находит ли текстовый запрос то, что просили.

In [ ]:
def show(rows, title, n=8, caption=None):
    rows = rows[:n]
    fig, axes = plt.subplots(1, n, figsize=(2.6 * n, 2.8))
    for ax in axes:
        ax.axis("off")
    for ax, (_, r) in zip(axes, rows.iterrows()):
        ax.imshow(open_rgb(PHOTOS_ROOT / r["path"], max_side=400))
        ax.set_title(caption(r) if caption else r["listing_id"], fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

rng = np.random.RandomState(0)
for room in ["kitchen", "living_room", "bedroom", "bathroom", "exterior", "floor_plan", "other"]:
    sub = rooms[rooms.room_type == room]
    if len(sub):
        show(sub.sample(min(8, len(sub)), random_state=0), f"{room}: {len(sub)} фото",
             caption=lambda r: f"{r.room_conf:.2f}")

show(rooms.sort_values("render_prob", ascending=False), "самые «рендерные» — реально рендеры?",
     caption=lambda r: f"render {r.render_prob:.2f}")

In [ ]:
QUERIES = [
    ("светлая кухня с белыми фасадами", ["kitchen"]),
    ("старый советский ремонт, ковёр на стене", None),
    ("современная гостиная в скандинавском стиле", ["living_room"]),
    ("ванная с тёмной плиткой под мрамор", ["bathroom"]),
    ("квартира без ремонта, голые стены", None),
]
room_arr, render_arr, lid_arr = rooms.room_type.to_numpy(), rooms.render_prob.to_numpy(), idx.listing_id.to_numpy()

for text, want_rooms in QUERIES:
    s = E.astype(np.float32) @ embed_texts([text])[0]
    mask = valid & (render_arr < RENDER_THR)
    if want_rooms:
        mask &= np.isin(room_arr, want_rooms)
    s[~mask] = -np.inf
    order, seen, top = np.argsort(-s), set(), []
    for i in order:                      # одно фото на объявление, как в search.py
        if lid_arr[i] in seen:
            continue
        seen.add(lid_arr[i]); top.append(i)
        if len(top) == 8:
            break
    show(idx.iloc[top].assign(score=s[top]), f"«{text}»", caption=lambda r: f"{r.listing_id}\n{r.score:.3f}")

### Поиск по фото

То же пространство, только вектор запроса даёт не текст, а картинка. Первое фото в ряду —
запрос, дальше похожие из других объявлений. Так же будет работать сгенерированная ИИ картинка.

In [ ]:
rs = np.random.RandomState(7)
pool = np.flatnonzero(valid & np.isin(room_arr, list(INTERIOR)) & (render_arr < RENDER_THR))
for qi in rs.choice(pool, 3, replace=False):
    s = E.astype(np.float32) @ E[qi].astype(np.float32)
    mask = valid & (render_arr < RENDER_THR) & (lid_arr != lid_arr[qi])   # своё же объявление не показываем
    mask &= room_arr == room_arr[qi]                                       # ищем ту же комнату
    s[np.where(~mask)] = -np.inf
    order, seen, top = np.argsort(-s), set(), []
    for i2 in order:
        if lid_arr[i2] in seen:
            continue
        seen.add(lid_arr[i2]); top.append(i2)
        if len(top) == 7:
            break
    show(pd.concat([idx.iloc[[qi]].assign(score=1.0), idx.iloc[top].assign(score=s[top])]),
         f"поиск по фото: {idx.path[qi]} ({room_arr[qi]})", n=8,
         caption=lambda r: f"{r.listing_id}\n{r.score:.3f}")

## Дальше — локально

1. Скачай из Output папки `artifacts/` и `data/` -> положи в `krisha_search/`.
2. Фото для показа результатов: распакуй `KrishaParser/data/photos` в `krisha_search/data/photos`
   (системный `unzip` на Mac этот zip64 не читает — через Python `zipfile`).
3. `python run_pipeline.py prepare` -> `python run_pipeline.py status` -> `search`.